# Task 6: Customer Segmentation — Mall Customer Dataset

**Synent Technologies — Data Science Internship Program**

**Author:** Akshitha K

---

## Objective
Group mall customers into meaningful segments based on their purchasing
behavior, using **K-Means clustering** on annual income and spending
score. The goal is to go beyond a generic clustering exercise and produce
**actionable, business-relevant customer personas** that a marketing team
could actually use.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

sns.set_style('whitegrid')

## 2. Load and Inspect the Dataset

In [ ]:
df = pd.read_csv('Mall_Customers.csv')
df.head()

In [ ]:
print('Shape:', df.shape)
df.info()
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

The dataset is already clean — **200 rows, 5 columns, zero missing values,
zero duplicates**. No imputation or deduplication is required.

## 3. Data Preprocessing

In [ ]:
# Rename columns to clean, consistent snake_case
df = df.rename(columns={
    'CustomerID': 'customer_id',
    'Gender': 'gender',
    'Age': 'age',
    'Annual Income (k$)': 'annual_income_k',
    'Spending Score (1-100)': 'spending_score'
})
df.head()

## 4. Exploratory Data Analysis

Before clustering, understand the distributions and relationships in the
raw features.

In [ ]:
df[['age','annual_income_k','spending_score']].describe().round(1)

In [ ]:
print('Gender distribution:\n', df['gender'].value_counts())
print('\nPercent female:', round((df["gender"]=="Female").mean()*100, 1), '%')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,4))
sns.histplot(df['age'], bins=15, ax=axes[0], color='#4C72B0')
axes[0].set_title('Age Distribution')
sns.histplot(df['annual_income_k'], bins=15, ax=axes[1], color='#55A868')
axes[1].set_title('Annual Income Distribution (k$)')
sns.histplot(df['spending_score'], bins=15, ax=axes[2], color='#C44E52')
axes[2].set_title('Spending Score Distribution')
plt.tight_layout()
plt.show()

**Observation:** Income is right-skewed with most customers earning
40–80k$, while spending score is fairly evenly spread from 1–99 — meaning
income and spending behavior are not simply correlated with each other, and
each customer's spending score depends on more than income alone. This is
exactly the kind of pattern clustering is well suited to uncover.

## 5. Apply K-Means Clustering

We cluster on **annual income** and **spending score** — the two features
that most directly describe purchasing behavior. Features are standardized
before clustering since K-Means is distance-based.

In [ ]:
X = df[['annual_income_k', 'spending_score']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 5.1 Determine Optimal k — Elbow Method + Silhouette Score

In [ ]:
wcss, sil_scores = [], []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))
wcss_full = [KMeans(n_clusters=1, random_state=42, n_init=10).fit(X_scaled).inertia_] + wcss

fig, axes = plt.subplots(1, 2, figsize=(12,5))
axes[0].plot(range(1,11), wcss_full, marker='o', color='#4C72B0')
axes[0].axvline(5, color='red', linestyle='--', alpha=0.6, label='Chosen k=5')
axes[0].set_title('Elbow Method — WCSS vs k', fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)')
axes[0].set_ylabel('WCSS (Inertia)')
axes[0].legend()

axes[1].plot(list(K_range), sil_scores, marker='o', color='#55A868')
axes[1].axvline(5, color='red', linestyle='--', alpha=0.6, label='Chosen k=5')
axes[1].set_title('Silhouette Score vs k', fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].legend()
plt.tight_layout()
plt.savefig('visualizations/chart1_elbow_silhouette.png', dpi=120)
plt.show()

print('Silhouette scores by k:', dict(zip(K_range, [round(s,4) for s in sil_scores])))

**Observation:** WCSS drops sharply up to k=5 and flattens after that
(elbow point), and the silhouette score **peaks at k=5 (0.555)** — the
highest of any tested value. Both methods agree: **k=5 is the optimal
number of clusters**, and a score of 0.555 indicates reasonably
well-separated, cohesive clusters.

### 5.2 Fit Final Model (k=5)

In [ ]:
kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)
centroids_original = scaler.inverse_transform(kmeans.cluster_centers_)

final_silhouette = silhouette_score(X_scaled, df['cluster'])
print('Final silhouette score (k=5):', round(final_silhouette, 4))

## 6. Visualize the Clusters

In [ ]:
cluster_names = {
    0: 'Average Customers',
    1: 'High Income / High Spending (Target)',
    2: 'Low Income / High Spending (Impulsive)',
    3: 'High Income / Low Spending (Frugal)',
    4: 'Low Income / Low Spending (Budget-Conscious)'
}
df['segment'] = df['cluster'].map(cluster_names)

palette = sns.color_palette('Set2', 5)
plt.figure(figsize=(8,6))
for c in sorted(df['cluster'].unique()):
    subset = df[df['cluster']==c]
    plt.scatter(subset['annual_income_k'], subset['spending_score'],
                label=cluster_names[c], color=palette[c], alpha=0.75, edgecolor='k', linewidth=0.3)
plt.scatter(centroids_original[:,0], centroids_original[:,1], marker='X', s=250, c='black', label='Centroids')
plt.title('Customer Segments: Annual Income vs Spending Score', fontsize=13, fontweight='bold')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(bbox_to_anchor=(1.02,1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig('visualizations/chart2_cluster_scatter.png', dpi=120)
plt.show()

## 7. Cluster Profiling

In [ ]:
profile = df.groupby('segment').agg(
    count=('customer_id','count'), avg_age=('age','mean'),
    avg_income=('annual_income_k','mean'), avg_spending=('spending_score','mean')
).round(1).reindex([cluster_names[i] for i in range(5)])
profile

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15,5))
metrics = ['avg_age','avg_income','avg_spending']
titles = ['Average Age', 'Average Annual Income (k$)', 'Average Spending Score']
for ax, m, t in zip(axes, metrics, titles):
    bars = ax.bar(range(5), profile[m], color=palette)
    ax.set_xticks(range(5))
    ax.set_xticklabels([f'Seg {i}' for i in range(5)])
    ax.set_title(t, fontweight='bold')
    for i, v in enumerate(profile[m]):
        ax.text(i, v+0.5, str(v), ha='center', fontsize=9)
plt.suptitle('Cluster Profile Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('visualizations/chart3_cluster_profile.png', dpi=120)
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,5))
gender_ct = pd.crosstab(df['cluster'], df['gender'])
gender_ct.plot(kind='bar', stacked=True, ax=axes[0], color=['#C44E52','#4C72B0'])
axes[0].set_title('Gender Distribution by Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

sns.boxplot(data=df, x='cluster', y='age', ax=axes[1], hue='cluster', palette=palette, legend=False)
axes[1].set_title('Age Distribution by Cluster', fontweight='bold')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Age')
plt.tight_layout()
plt.savefig('visualizations/chart4_gender_age_by_cluster.png', dpi=120)
plt.show()

## 8. Export Results

In [ ]:
df.to_csv('mall_customers_segmented.csv', index=False)
profile.to_csv('cluster_profile_summary.csv')
print('Saved: mall_customers_segmented.csv, cluster_profile_summary.csv')

## 9. Key Insights

- **Segment 1 — High Income / High Spending (Target, n=39):** The most
  valuable segment — avg. income $86.5k, avg. spending score 82.1. These
  are the customers premium offers and loyalty programs should prioritize.
- **Segment 3 — High Income / Low Spending (Frugal, n=35):** Avg. income
  $88.2k but spending score only 17.1 — the single biggest **untapped
  opportunity**. They can clearly afford to spend more; targeted
  promotions or personalized outreach could convert this segment.
- **Segment 2 — Low Income / High Spending (Impulsive, n=22):** Avg.
  income just $25.7k but spending score 79.4 — the youngest segment
  (avg. age 25.3) and likely driven by impulse or trend-based purchases
  rather than disposable income.
- **Segment 4 — Low Income / Low Spending (Budget-Conscious, n=23):**
  Lowest spending behavior, oldest average age (45.2) alongside Segment 0.
  Price-sensitive; best served by discounts and value bundles.
- **Segment 0 — Average Customers (n=81):** The largest segment (40.5% of
  all customers) sits squarely in the middle on both income and spending —
  a broad, general-purpose audience.
- **Income and spending score are not correlated with gender in any
  cluster** — every segment is a fairly even gender mix, so gender is not
  a useful variable for this particular segmentation.
- **k=5 was statistically validated**, not guessed — both the elbow curve
  and silhouette score (peaking at 0.555) independently pointed to the
  same value.

## 10. Conclusion

K-Means clustering on annual income and spending score revealed five
statistically distinct and business-meaningful customer segments. Rather
than treating all 200 customers as one group, this segmentation uncovers
a clear actionable finding: the "High Income / Low Spending" segment
represents real revenue upside, since these customers have the means to
spend significantly more than they currently do. This is the kind of
unsupervised-learning-to-business-insight pipeline that demonstrates
practical, portfolio-ready data science skill beyond basic EDA.